In [ ]:
data_path: str = None

In [ ]:
import mutopia.analysis as mu
import mutopia.plot.track_plot as tr
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import os

In [ ]:
figure_dir = os.path.join(os.path.dirname(data_path), "figures")
if not os.path.exists(figure_dir):
    os.makedirs(figure_dir, exist_ok=True)

In [ ]:
data = mu.gt.load_dataset(data_path)

In [ ]:
mu.pl.plot_signature_panel(data)
plt.savefig(os.path.join(figure_dir, "signature_panel.png"), dpi=300, bbox_inches='tight')

In [ ]:
with PdfPages(os.path.join(figure_dir, "interaction_matrices.pdf")) as pdf:
    for component in mu.gt.list_components(data):
        fig = mu.pl.plot_interaction_matrix(data, component)
        pdf.savefig(fig, bbox_inches='tight')

In [ ]:
contribs = data["contributions"]
data["relative_contributions"] = contribs.squeeze()/contribs.squeeze().sum(dim="component")
_, ax = plt.subplots(figsize=(4,6))
sns.boxenplot(
    data["relative_contributions"].to_pandas().melt(ignore_index=False),
    x="value",
    y="component",
    alpha=0.5,
    ax = ax,
)
ax.set(xlim=(0,0.5), xlabel="Fraction of sample", ylabel="Mutational process")
sns.despine()
plt.savefig(os.path.join(figure_dir, "relative_contributions.png"), dpi=300, bbox_inches='tight')

In [ ]:
g = sns.clustermap(data["relative_contributions"].to_pandas().T, cbar_pos=None, figsize=(8,8), cmap="viridis", vmax=0.35, dendrogram_ratio=(0.05,0.2))
g.ax_heatmap.set_xlabel("Sample")
g.ax_heatmap.set_xticklabels([])
plt.savefig(os.path.join(figure_dir, "exposures_matrix.png"), dpi=300, bbox_inches='tight')

In [ ]:
data["relative_contributions"].to_pandas().to_csv(os.path.join(figure_dir, "relative_contributions.csv"))

In [ ]:
data.attrs["regions_file"] = os.path.join("../../../../gtensors/", os.path.basename(data.attrs["regions_file"]))

In [ ]:
view = tr.make_view(data, "chr2:1000000-85000000")

In [ ]:
prediction_config = lambda view, scalebar_bp=1_000_000 : (
    tr.scale_bar(scalebar_bp, scale="mb" if scalebar_bp >= 1_000_000 else "kb"),
    tr.ideogram("annotations/hg38.cytoband.bed.gz"),
    tr.tracks.plot_marginal_observed_vs_expected(
        view,
        pred_smooth=5,
        smooth=5,
        height=1.25,
    ),
)

tr.plot_view(prediction_config, view, width=10)
plt.savefig(os.path.join(figure_dir, "observed_vs_prediction.png"), dpi=300, bbox_inches='tight')

In [ ]:
topography = tr.TopographyTransformer().fit(data)

topo_config = lambda view, scalebar_bp=1_000_000 : (
    prediction_config(view, scalebar_bp),
    tr.plot_topography(topography, height=2),
)
tr.plot_view(topo_config, view, width=10)
plt.savefig(os.path.join(figure_dir, "topography.png"), dpi=300, bbox_inches='tight')

In [ ]:
import numpy as np
from functools import partial

def _plot_component(comp_name, ax):
    ax = mu.pl.plot_spectrum(mu.gt.fetch_component(data, comp_name).sel(genome_state="Baseline"), ax=ax)
    ax.set_ylabel(
        comp_name, fontsize=9, rotation=0, labelpad=10, va="center", ha="right"
    )
    return ax

def _plot_shared_effects(component_name, ax):
    shared_effect = (
        data.sections["Spectra"]["shared_effects"]
        .sel(component=component_name)
        .to_pandas()
        .iloc[1:,]
        .to_frame()
        .T
    )

    sns.heatmap(
        shared_effect,
        cmap=mu.pl.diverging_palette,
        linewidths=0.5,
        vmin=-0.5,
        vmax=0.5,
        cbar=False,
        ax=ax,
        square=True,
    )
    ax.set(
        ylabel="",
        xlabel="",
        yticklabels=[],
    )
    ax.set_xticks([0.5, 1.5, 2.5, 3.5])
    ax.set_yticks([])
    # remove the xticks but keep the labels

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("lightgrey")
        spine.set_linewidth(0.5)
    ax.set_xticklabels(shared_effect.columns.str.removesuffix("-centered").str.replace(":", " | "), rotation=90, fontsize=7)

phases = ["G1b", "S1", "S2", "S3", "S4", "G2"]
repliseq = tr.heatmap_plot(
    tr.pipeline(
        tr.feature_matrix(*["Repliseq" + phase for phase in phases]),
        tr.apply_rows(tr.minmax_scale),
    ),
    palette="crest_r",
    cbar=False,
    label="Cell cycle\nphase",
    ax_fn=lambda ax: (ax.set_yticklabels(reversed(phases))),
)

components_config = lambda view, scalebar_bp=1_000_000 : (
    tr.columns(..., tr.scale_bar(scalebar_bp, scale="mb" if scalebar_bp >= 1_000_000 else "kb"), ..., height=0.1),
    tr.columns(..., tr.ideogram("annotations/hg38.cytoband.bed.gz"), ..., height=0.1,),
    tr.columns(..., tr.tracks.plot_marginal_observed_vs_expected(
        view,
        pred_smooth=7,
        smooth=7,
        height=1.25,
    ), ...),
    tr.columns(..., tr.plot_topography(topography), ..., height=2),
    tr.columns(..., tr.text_banner("Genomic features"), ..., height=0.35),
    tr.columns(..., repliseq, ..., height=1),
    tr.columns(..., tr.heatmap_plot(
        tr.pipeline(
            tr.feature_matrix("GeneExpression"),
            lambda x : x.expand_dims("feature"),
            lambda x : np.log1p(x),
        ),
        palette="Greys",
        ax_fn=lambda ax: (
            ax.spines["top"].set_visible(False),
            ax.spines["right"].set_visible(False),
            ax.spines["left"].set_visible(False),
            ax.spines["bottom"].set_visible(False),
            ax.set_yticklabels([]),
        ),
        label="Gene Expression",
        cbar=False,
    ), ..., height=0.2),
    tr.columns(tr.text_banner("Spectra"), tr.text_banner("Mutation rate"), tr.text_banner("Strand effect"), height=0.5),
    *[
        tr.columns(
            tr.custom_plot(partial(_plot_component, component)), 
            track,
            tr.custom_plot(partial(_plot_shared_effects, component)),
            height=0.5
        )
        for component in tr.order_components(data)
        for track in tr.plot_component_rates(view, component, label=" ", smooth=20)
    ]
)
tr.plot_view(components_config, view, width_ratios=[4,8,1], width=11)
plt.savefig(os.path.join(figure_dir, "summary_vew.png"), dpi=300, bbox_inches='tight')